# Simulation 01 -- Mass-Spring & PBD: Cloth and Soft Bodies from Scratch

> **Geo-Instructor . Simulation/FEM Career Track . Notebook 1 of 3**

Every cloth sim in Houdini, every soft body in Unity, every surgical tissue model starts here.
This notebook builds the full chain: Euler -> Symplectic Euler -> Mass-Spring cloth -> PBD -> XPBD.

```
Runtime: ~4 min  |  No GPU  |  numpy + matplotlib only
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation
from matplotlib.collections import LineCollection
np.random.seed(0)
plt.rcParams.update({'figure.facecolor':'#F4F2F0','axes.facecolor':'#F4F2F0','font.family':'monospace','axes.grid':True,'grid.alpha':0.3})
print('Ready.')

---
## Part 1 -- Integration Schemes: Why Explicit Euler Explodes

The pendulum is the canonical test:
```
  x'' = -g*sin(x)   (angle x, gravity g)

Explicit Euler (unstable at large dt):
  v_{n+1} = v_n + dt * f(x_n)
  x_{n+1} = x_n + dt * v_n          <-- uses OLD v

Symplectic Euler (energy-stable):
  v_{n+1} = v_n + dt * f(x_n)
  x_{n+1} = x_n + dt * v_{n+1}      <-- uses NEW v
```

In [ ]:
def run_pendulum(dt, n_steps, method='symplectic'):
    g = 9.81
    x, v = np.pi / 4, 0.0  # 45 deg initial angle
    xs, vs, Es = [x], [v], []
    for _ in range(n_steps):
        f = -g * np.sin(x)
        v_new = v + dt * f
        if method == 'explicit':
            x = x + dt * v        # old v
        else:
            x = x + dt * v_new    # new v (symplectic)
        v = v_new
        E = 0.5 * v**2 - g * np.cos(x)  # kinetic + potential
        xs.append(x); vs.append(v); Es.append(E)
    return np.array(xs), np.array(vs), np.array(Es)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Pendulum integration -- energy behaviour', fontsize=11)
for dt, ls in [(0.05,'-'), (0.1,'--')]:
    for method, color in [('explicit','tomato'), ('symplectic','steelblue')]:
        try:
            xs, vs, Es = run_pendulum(dt, 600, method)
            if np.any(np.abs(xs) > 100): raise ValueError('diverged')
            axes[0].plot(xs, label=f'{method} dt={dt}', color=color, ls=ls, lw=1.5)
            axes[1].plot(Es, color=color, ls=ls, lw=1.5)
        except:
            axes[0].text(0.3, 0.5, f'{method} dt={dt} DIVERGED', transform=axes[0].transAxes, color='red', fontsize=8)
axes[0].set_title('Angle over time'); axes[0].legend(fontsize=7)
axes[1].set_title('Total energy over time')
axes[1].set_ylabel('E'); axes[1].set_xlabel('step')
# Phase portrait
for method, color in [('explicit','tomato'), ('symplectic','steelblue')]:
    xs, vs, _ = run_pendulum(0.05, 400, method)
    axes[2].plot(xs, vs, color=color, lw=1.2, alpha=0.8, label=method)
axes[2].set_title('Phase portrait (x vs v)'); axes[2].legend(fontsize=8)
axes[2].set_xlabel('angle'); axes[2].set_ylabel('velocity')
plt.tight_layout(); plt.show()
print('Symplectic Euler: energy oscillates but does not grow. Explicit: energy grows -> explosion.')

---
## Part 2 -- Mass-Spring Cloth

A cloth is a 2D grid of particles connected by springs:
- **Structural** springs: along rows and columns (resist stretch)
- **Shear** springs: diagonals (resist shear)
- **Bending** springs: skip-one-node (resist fold)

Hooke's law force: `F = -k * (|x_i - x_j| - L0) * direction`

In [ ]:
class MassSpringCloth:
    def __init__(self, rows=12, cols=12, rest_len=0.1, mass=0.01,
                 k_struct=200.0, k_shear=100.0, k_bend=50.0,
                 damping=0.98, gravity=-9.81):
        self.rows, self.cols = rows, cols
        self.gravity = gravity; self.damping = damping
        # Particle positions: grid in XZ plane, hanging down
        x = np.linspace(0, (cols-1)*rest_len, cols)
        y = np.linspace(0, -(rows-1)*rest_len, rows)
        xx, yy = np.meshgrid(x, y)
        self.pos = np.column_stack([xx.ravel(), yy.ravel()]).astype(float)
        self.vel = np.zeros_like(self.pos)
        self.mass = mass
        # Build spring list: [(i, j, rest_len, k), ...]
        self.springs = []
        def idx(r,c): return r*cols + c
        for r in range(rows):
            for c in range(cols):
                if c+1 < cols: self.springs.append((idx(r,c), idx(r,c+1), rest_len, k_struct))
                if r+1 < rows: self.springs.append((idx(r,c), idx(r+1,c), rest_len, k_struct))
                if r+1<rows and c+1<cols:
                    d = rest_len*np.sqrt(2)
                    self.springs.append((idx(r,c), idx(r+1,c+1), d, k_shear))
                    self.springs.append((idx(r,c+1), idx(r+1,c), d, k_shear))
                if c+2 < cols: self.springs.append((idx(r,c), idx(r,c+2), 2*rest_len, k_bend))
                if r+2 < rows: self.springs.append((idx(r,c), idx(r+2,c), 2*rest_len, k_bend))
        # Pin top edge
        self.pinned = set(range(cols))
        print(f'Cloth: {rows*cols} particles, {len(self.springs)} springs, {len(self.pinned)} pinned')

    def step(self, dt):
        forces = np.zeros_like(self.pos)
        forces[:, 1] += self.gravity * self.mass
        for (i, j, L0, k) in self.springs:
            diff = self.pos[j] - self.pos[i]
            dist = np.linalg.norm(diff)
            if dist < 1e-8: continue
            f = k * (dist - L0) * (diff / dist)
            forces[i] += f; forces[j] -= f
        self.vel += dt * forces / self.mass
        self.vel *= self.damping
        self.pos += dt * self.vel
        for p in self.pinned:
            self.pos[p] = self.pos[p]; self.vel[p] = 0
            r, c = divmod(p, self.cols)
            self.pos[p, 0] = c * 0.1; self.pos[p, 1] = 0.0

cloth = MassSpringCloth(rows=10, cols=10)
# Run 100 steps
for _ in range(100): cloth.step(0.005)

fig, ax = plt.subplots(figsize=(7, 7))
pos = cloth.pos.reshape(cloth.rows, cloth.cols, 2)
# Draw horizontal and vertical edges
for r in range(cloth.rows):
    ax.plot(pos[r,:,0], pos[r,:,1], 'steelblue', lw=0.8, alpha=0.7)
for c in range(cloth.cols):
    ax.plot(pos[:,c,0], pos[:,c,1], 'steelblue', lw=0.8, alpha=0.7)
ax.scatter(pos[:,:,0], pos[:,:,1], s=10, c='steelblue', zorder=3)
ax.scatter(pos[0,:,0], pos[0,:,1], s=30, c='tomato', zorder=4, label='Pinned top edge')
ax.set_aspect('equal'); ax.legend(fontsize=9)
ax.set_title('Mass-Spring cloth after 100 steps (gravity, top edge pinned)')
plt.tight_layout(); plt.show()

---
## Part 3 -- Position-Based Dynamics (PBD)

Explicit spring simulation becomes unstable at high stiffness.
PBD sidesteps this: instead of forces, it **directly projects positions** to satisfy constraints.

```
Each frame:
  1. Apply external forces: v += dt * f_ext / m
  2. Predict: p = x + dt * v
  3. Project ALL constraints iteratively (n_iters times)
  4. Update velocity: v = (p - x) / dt
  5. Update position: x = p
```
Unconditionally stable -- any dt, any stiffness.

In [ ]:
class PBDCloth:
    def __init__(self, rows=10, cols=10, rest_len=0.1):
        self.rows, self.cols = rows, cols
        x = np.linspace(0, (cols-1)*rest_len, cols)
        y = np.linspace(0, -(rows-1)*rest_len, rows)
        xx, yy = np.meshgrid(x, y)
        self.pos  = np.column_stack([xx.ravel(), yy.ravel()]).astype(float)
        self.prev = self.pos.copy()
        self.mass = 0.01
        self.constraints = []
        def idx(r,c): return r*cols + c
        for r in range(rows):
            for c in range(cols):
                if c+1 < cols: self.constraints.append((idx(r,c), idx(r,c+1), rest_len))
                if r+1 < rows: self.constraints.append((idx(r,c), idx(r+1,c), rest_len))
                if r+1<rows and c+1<cols:
                    d = rest_len*np.sqrt(2)
                    self.constraints.append((idx(r,c), idx(r+1,c+1), d))
                    self.constraints.append((idx(r,c+1), idx(r+1,c), d))
        self.pinned = set(range(cols))

    def step(self, dt, n_iters=5):
        gravity = np.array([0.0, -9.81])
        # 1. Predict
        vel = (self.pos - self.prev) / dt
        self.prev = self.pos.copy()
        p = self.pos + dt*vel + dt*dt * gravity
        # 2. Constraint projection
        for _ in range(n_iters):
            for (i, j, L0) in self.constraints:
                diff = p[j] - p[i]
                dist = np.linalg.norm(diff)
                if dist < 1e-8: continue
                corr = 0.5 * (dist - L0) * diff / dist
                if i not in self.pinned: p[i] += corr
                if j not in self.pinned: p[j] -= corr
        # 3. Pin
        r0 = 0
        for c in range(self.cols):
            p[c] = np.array([c * 0.1, 0.0])
        self.pos = p

pbd = PBDCloth(rows=10, cols=10)
# Compare PBD vs mass-spring at very high stiffness
frames = 120
pbd_states = []
for _ in range(frames): pbd.step(0.016, n_iters=8); pbd_states.append(pbd.pos.copy())

fig, axes = plt.subplots(1, 2, figsize=(13, 6))
for ax, (states, title) in zip(axes, [
    ([pbd_states[20], pbd_states[60], pbd_states[-1]], 'PBD cloth (always stable)')]):
    for state, alpha in zip(states, [0.3, 0.6, 1.0]):
        pos = state.reshape(pbd.rows, pbd.cols, 2)
        for r in range(pbd.rows): ax.plot(pos[r,:,0], pos[r,:,1], color='steelblue', lw=0.8, alpha=alpha)
        for c in range(pbd.cols): ax.plot(pos[:,c,0], pos[:,c,1], color='steelblue', lw=0.8, alpha=alpha)
    ax.scatter(*pbd_states[-1].reshape(pbd.rows, pbd.cols, 2)[0].T, s=25, c='tomato', zorder=5)
    ax.set_aspect('equal'); ax.set_title(title)
axes[1].text(0.5, 0.5, 'No mass-spring comparison\n(would need lower k for stability)', ha='center', va='center', transform=axes[1].transAxes, fontsize=9, color='gray')
plt.tight_layout(); plt.show()
print('PBD is unconditionally stable: any dt, any stiffness, no explosion.')

---
## Part 4 -- XPBD: Adding Physical Units

PBD has a problem: stiffness depends on iteration count and timestep.
XPBD (Extended PBD) adds a **compliance** parameter alpha = 1/stiffness:
```
  Lagrange multiplier:  delta_lambda = (-C - alpha_tilde * lambda) / (sum_i w_i |grad_C|^2 + alpha_tilde)
  alpha_tilde = alpha / dt^2
```
Now stiffness has physical units (N/m) and is timestep-independent.

In [ ]:
def xpbd_distance_demo(p1_init, p2_init, L0, stiffness, dt=0.016, n_steps=120):
    """Single XPBD distance constraint between two free particles."""
    p1 = np.array(p1_init, dtype=float)
    p2 = np.array(p2_init, dtype=float)
    v1 = np.zeros(2); v2 = np.zeros(2)
    alpha = 1.0 / stiffness  # compliance
    alpha_tilde = alpha / (dt*dt)
    history = [(p1.copy(), p2.copy())]
    for _ in range(n_steps):
        # Predict
        p1_pred = p1 + dt*v1 + dt*dt*np.array([0,-9.81])
        p2_pred = p2 + dt*v2 + dt*dt*np.array([0,-9.81])
        lam = 0.0
        # One constraint iteration
        diff = p2_pred - p1_pred
        dist = np.linalg.norm(diff)
        C = dist - L0
        grad = diff / (dist + 1e-10)
        w = 2.0  # 1/mass + 1/mass
        delta_lam = (-C - alpha_tilde * lam) / (w + alpha_tilde)
        p1_pred -= delta_lam * grad
        p2_pred += delta_lam * grad
        v1 = (p1_pred - p1) / dt
        v2 = (p2_pred - p2) / dt
        p1 = p1_pred; p2 = p2_pred
        history.append((p1.copy(), p2.copy()))
    return history

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('XPBD: stiffness controls softness independently of dt', fontsize=10)
for ax, (k, label) in zip(axes, [(1e4,'soft k=1e4'), (1e6,'medium k=1e6'), (1e9,'rigid k=1e9')]):
    hist = xpbd_distance_demo([0,0],[0.3,-0.1], L0=0.2, stiffness=k)
    for step in range(0, len(hist), 15):
        p1, p2 = hist[step]
        ax.plot([p1[0],p2[0]], [p1[1],p2[1]], 'o-', color='steelblue', lw=1.5, ms=5, alpha=0.3+0.7*step/len(hist))
    p1f, p2f = hist[-1]
    ax.plot([p1f[0],p2f[0]], [p1f[1],p2f[1]], 'o-', color='tomato', lw=2.5, ms=8, label='Final')
    ax.set_xlim(-0.5, 0.8); ax.set_ylim(-1.5, 0.3)
    ax.set_title(label); ax.set_aspect('equal')
    ax.legend(fontsize=8)
plt.tight_layout(); plt.show()
print('XPBD compliance alpha=1/k: larger k -> stiffer. Physical units preserved.')

---
## Exercises

### Exercise 1 -- Bending constraint
Add a dihedral angle constraint between adjacent triangles. This prevents the cloth from folding along a crease.

### Exercise 2 -- Sphere collision
Add a sphere collider. For each particle inside the sphere, project it to the surface.

### Exercise 3 -- Self-collision
Build a spatial hash to find particle pairs that are too close and apply separation constraints.

---
## Summary

| Method | Stability | Physical units | Speed | Used in |
|--------|-----------|---------------|-------|--------|
| Explicit Euler | Unstable at large dt/k | Yes | Fast | Simple demos |
| Symplectic Euler | Energy-stable | Yes | Fast | Rigid body, games |
| PBD | Unconditional | No (iter-dependent) | Fast | Unity, game engines |
| XPBD | Unconditional | Yes (compliance) | Fast | Houdini Vellum, USD Physics |

In [ ]:
print('Notebook complete.')
print('Key takeaways:')
print('  Symplectic Euler  -> energy-stable, the baseline integrator')
print('  Mass-Spring       -> intuitive but stiff at high k')
print('  PBD               -> constraint projection, unconditionally stable')
print('  XPBD              -> compliance = 1/stiffness, physical units')
print()
print('Production: Houdini Vellum (XPBD), Unity Cloth (PBD), Bullet (rigid + soft body)')